#  SkinSight AI — 05 · Recommandations & démo pipeline

Ce notebook démontre le pipeline complet end-to-end :
upload d'une image → prédiction → recommandations personnalisées + routine skincare.

**C'est la démo à montrer en soutenance.**

In [1]:
import os, joblib, io
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

BASE_DIR  = os.path.abspath(os.path.join(os.getcwd(), '..'))
MODEL_DIR = os.path.join(BASE_DIR, 'models')
DATA_DIR  = os.path.join(BASE_DIR, 'data')

CLASSES = ['saine', 'acne_inflammatoire', 'acne_non_inflammatoire', 'rosacee', 'hyperpigmentation']
CLASSES_FR = {
    'saine': 'Peau saine',
    'acne_inflammatoire': 'Acné inflammatoire',
    'acne_non_inflammatoire': 'Acné non inflammatoire',
    'rosacee': 'Rosacée',
    'hyperpigmentation': 'Hyperpigmentation'
}
COLORS = ['#C07A5A', '#8B6A8B', '#D4A896', '#7A9E7E', '#B8956A']
print('Environnement prêt')

Environnement prêt


## 1. Chargement du pipeline

In [2]:
from tensorflow.keras.applications import MobileNetV2, DenseNet121, InceptionV3
from tensorflow.keras.applications.mobilenet_v2  import preprocess_input as pm
from tensorflow.keras.applications.densenet       import preprocess_input as pd
from tensorflow.keras.applications.inception_v3   import preprocess_input as pi
from tensorflow.keras.preprocessing.image import img_to_array

print('Chargement des CNN...')
mob = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')
den = DenseNet121(weights='imagenet', include_top=False, pooling='avg')
inc = InceptionV3(weights='imagenet', include_top=False, pooling='avg', input_shape=(299,299,3))
print('   MobileNetV2 | DenseNet121 | InceptionV3')

pca  = joblib.load(os.path.join(MODEL_DIR, 'pca_512.pkl'))
lgbm = joblib.load(os.path.join(MODEL_DIR, 'lightgbm.pkl'))
print('   PCA-512 | LightGBM')
print('Pipeline prêt !')

Chargement des CNN...
   MobileNetV2 | DenseNet121 | InceptionV3
   PCA-512 | LightGBM
Pipeline prêt !


## 2. Fonction de prédiction

In [3]:
def predict_image(image_path: str) -> dict:
    """Pipeline complet : image → diagnostic + scores."""
    img = Image.open(image_path).convert('RGB')

    def prep(size, fn):
        arr = img_to_array(img.resize(size))
        return fn(np.expand_dims(arr, 0))

    feats = np.concatenate([
        mob.predict(prep((224,224), pm), verbose=0).flatten(),
        den.predict(prep((224,224), pd), verbose=0).flatten(),
        inc.predict(prep((299,299), pi), verbose=0).flatten(),
    ]).reshape(1, -1)

    reduced = pca.transform(feats)
    proba   = lgbm.predict_proba(reduced)[0]
    idx     = np.argmax(proba)

    return {
        'pathologie':    CLASSES[idx],
        'pathologie_fr': CLASSES_FR[CLASSES[idx]],
        'confiance':     round(float(proba[idx]) * 100, 1),
        'scores':        {CLASSES_FR[c]: round(float(p)*100, 1) for c, p in zip(CLASSES, proba)},
        'image':         img,
    }

print('Fonction predict_image() définie')

Fonction predict_image() définie


## 3. Démo — tester sur une image

In [4]:
# ── Modifier ce chemin avec votre image de test ──────────────────────────────
import glob

# Chercher automatiquement une image de test dans val/
test_images = []
for cls in CLASSES:
    folder = os.path.join(DATA_DIR, 'val', cls)
    if os.path.exists(folder):
        imgs = glob.glob(os.path.join(folder, '*.jpg')) + glob.glob(os.path.join(folder, '*.png'))
        if imgs:
            test_images.append((cls, imgs[0]))

print(f'{len(test_images)} images de démonstration trouvées')
for cls, path in test_images:
    print(f'  {cls:<30} → {os.path.basename(path)}')

0 images de démonstration trouvées


In [5]:
import glob

test_images = []
for cls in CLASSES:
    # Chercher dans train/ au lieu de val/ (plus d'images)
    for split in ['val', 'train']:
        folder = os.path.join(BASE_DIR, 'data', split, cls)
        if os.path.exists(folder):
            imgs = glob.glob(os.path.join(folder, '*.jpg')) + \
                   glob.glob(os.path.join(folder, '*.jpeg')) + \
                   glob.glob(os.path.join(folder, '*.png'))
            if imgs:
                test_images.append((cls, imgs[0]))
                break

print(f'{len(test_images)} images trouvees')
for cls, path in test_images:
    print(f'  {cls:<30} -> {os.path.basename(path)}')

0 images trouvees


In [6]:
import os
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(os.listdir(os.path.join(BASE_DIR, 'data')))

['augmented', 'dataset_config.yaml', 'history.db', 'processed', 'raw', 'splits']


## 4. Affichage des scores de confiance

In [7]:
import glob

test_images = []
for cls in CLASSES:
    for split in ['val', 'test', 'train']:
        folder = os.path.join(BASE_DIR, 'data', 'splits', split, cls)
        if os.path.exists(folder):
            imgs = glob.glob(os.path.join(folder, '*.jpg')) + \
                   glob.glob(os.path.join(folder, '*.jpeg')) + \
                   glob.glob(os.path.join(folder, '*.png'))
            if imgs:
                test_images.append((cls, imgs[0]))
                break

print(f'{len(test_images)} images trouvées')
for cls, path in test_images:
    print(f'  {cls:<30} -> {os.path.basename(path)}')

5 images trouvées
  saine                          -> ISIC_0174527.jpg
  acne_inflammatoire             -> acne_inflammatoire_0002.jpg
  acne_non_inflammatoire         -> acne_non_inflammatoire_0002.jpg
  rosacee                        -> 07PerioralDermq.jpg
  hyperpigmentation              -> albinism-1.jpg


## 5. Recommandations générées

In [8]:
# Recréer result depuis la première image trouvée
cls_demo, img_path_demo = test_images[0]
result = predict_image(img_path_demo)

RECOMMENDATIONS = {
    'acne_inflammatoire':     ['Gel nettoyant acide salicylique 0.5%', 'Niacinamide 10%', 'Gel hydratant non-comédogène', 'SPF 50+'],
    'acne_non_inflammatoire': ['Gel moussant doux', 'BHA (acide salicylique 1%) 3×/semaine', 'Fluide léger non-comédogène', 'SPF 50+ fluide'],
    'rosacee':                ['Eau micellaire ultra-douce', 'Niacinamide 5%', 'Crème apaisante camomille/avoine', 'SPF 50+ minéral'],
    'hyperpigmentation':      ['Gel nettoyant doux', 'Vitamine C 10–20% matin', 'Crème légère compatible Vit C', 'SPF 50+ INDISPENSABLE'],
    'saine':                  ['Gel nettoyant doux 1×/jour', 'Sérum antioxydant Vit C', 'Crème adaptée au type de peau', 'SPF 30–50'],
}

patho = result['pathologie']
recs  = RECOMMENDATIONS.get(patho, [])

print(f'Diagnostic : {result["pathologie_fr"]} ({result["confiance"]}% de confiance)')
print()
print('Recommandations personnalisées :')
steps = ['① Nettoyage', '② Actif ciblé', '③ Hydratation', '④ Protection']
for step, rec in zip(steps, recs):
    print(f'  {step} -> {rec}')

print()
print('Ces recommandations sont informatives. Consulter un dermatologue pour un diagnostic clinique.')

Diagnostic : Peau saine (100.0% de confiance)

Recommandations personnalisées :
  ① Nettoyage -> Gel nettoyant doux 1×/jour
  ② Actif ciblé -> Sérum antioxydant Vit C
  ③ Hydratation -> Crème adaptée au type de peau
  ④ Protection -> SPF 30–50

Ces recommandations sont informatives. Consulter un dermatologue pour un diagnostic clinique.


## 6. Test via l'API FastAPI

In [9]:
import requests

# L'API doit tourner : uvicorn app.api.main:app --reload --port 8000
API_URL = 'http://localhost:8000'

try:
    # Test health
    r = requests.get(f'{API_URL}/', timeout=3)
    print(f'API status : {r.json()}')

    # Test predict
    img_path = test_images[0][1]
    with open(img_path, 'rb') as f:
        resp = requests.post(
            f'{API_URL}/predict',
            files={'file': ('test.jpg', f, 'image/jpeg')},
            timeout=120
        )
    data = resp.json()
    print(f"\nRésultat API /predict :")
    print(f"  Pathologie  : {data['pathologie_fr']}")
    print(f"  Confiance   : {data['confiance']}%")
    print(f"  Scores      : {data['scores']}")
    print(f"  Recommand.  : {len(data['recommandations'])} éléments")
    print(f"  Routine     : matin={len(data['routine']['matin'])} étapes | soir={len(data['routine']['soir'])} étapes")

except requests.exceptions.ConnectionError:
    print('  API non démarrée — lance uvicorn dans un autre terminal')
    print('   uvicorn app.api.main:app --reload --port 8000')

  API non démarrée — lance uvicorn dans un autre terminal
   uvicorn app.api.main:app --reload --port 8000


## Résumé du pipeline end-to-end

```
Image (JPG/PNG)
    ↓
MobileNetV2 + DenseNet121 + InceptionV3  →  4352D
    ↓
PCA (pca_512.pkl)                         →   512D
    ↓
LightGBM (lightgbm.pkl)                   →  Probabilités × 5 classes
    ↓
Diagnostic + Confiance + Recommandations + Routine skincare
    ↓
JSON → Frontend HTML/JS (app/index.html)
```

**Temps d'inférence moyen :** ~3–5s (CPU, chargement CNN inclus au 1er appel)

>  *SkinSight AI est un outil d'aide à la décision. Il ne remplace pas un avis médical professionnel.*